# 02 - Feature Engineering

## Objective

Prepare the healthcare dataset for future machine learning work.

This notebook will:

- Load the cleaned source dataset.
- Remove exact duplicate records after inspection.
- Convert date columns.
- Create useful admission-time features.
- Identify numerical and categorical columns.
- Save a processed dataset.

> **Important:** The dataset does not contain an explicit `Readmission` or `Readmitted` target. Therefore, this notebook does not create an artificial target or train a readmission classifier.


## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Define Dataset Paths

The notebook is stored inside:

```text
ml-training/notebooks/
```

The raw dataset is expected at:

```text
ml-training/dataset/raw/healthcare_dataset.csv
```


In [2]:
# Define project and dataset paths
PROJECT_ROOT = Path("..")

RAW_DATA_PATH = PROJECT_ROOT / "dataset" / "raw" / "healthcare_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
PROCESSED_DATA_PATH = PROCESSED_DIR / "healthcare_feature_engineered.csv"

print("Raw dataset path:", RAW_DATA_PATH.resolve())
print("Processed dataset path:", PROCESSED_DATA_PATH.resolve())

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {RAW_DATA_PATH}")


Raw dataset path: D:\hospital-readmission-project\ml-training\dataset\raw\healthcare_dataset.csv
Processed dataset path: D:\hospital-readmission-project\ml-training\dataset\processed\healthcare_feature_engineered.csv


## 3. Load the Dataset

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


## 4. Check Column Names and Data Types

In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())


Column names:
['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']

Data types:


Name                      str
Age                     int64
Gender                    str
Blood Type                str
Medical Condition         str
Date of Admission         str
Doctor                    str
Hospital                  str
Insurance Provider        str
Billing Amount        float64
Room Number             int64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object


Missing values:


Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

## 5. Remove Exact Duplicate Records

Exact duplicates are identified across all columns.

We first count and inspect them. In this preparation stage, exact duplicate rows are removed to prevent identical records from receiving unnecessary repeated weight during model training.

This does not resolve whether the dataset represents multiple legitimate hospital visits, because the dataset does not provide a reliable patient-visit identifier for that analysis.


In [5]:
duplicate_count_before = df.duplicated().sum()

print("Exact duplicate rows before removal:", duplicate_count_before)

if duplicate_count_before > 0:
    print("\nSample duplicate records:")
    display(
        df[df.duplicated(keep=False)]
        .sort_values(by=df.columns.tolist())
        .head(10)
    )


Exact duplicate rows before removal: 534

Sample duplicate records:


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
42407,ABIgaIL YOung,41,Female,O+,Hypertension,2022-12-15,Edward Kramer,Moore-Mcdaniel,UnitedHealthcare,1983.568297,192,Elective,2023-01-13,Ibuprofen,Normal
54285,ABIgaIL YOung,41,Female,O+,Hypertension,2022-12-15,Edward Kramer,Moore-Mcdaniel,UnitedHealthcare,1983.568297,192,Elective,2023-01-13,Ibuprofen,Normal
26025,ALIcia taYLoR,78,Male,O+,Asthma,2022-09-18,Dawn Burton,Wright LLC,Aetna,31465.274979,149,Elective,2022-10-15,Aspirin,Inconclusive
53104,ALIcia taYLoR,78,Male,O+,Asthma,2022-09-18,Dawn Burton,Wright LLC,Aetna,31465.274979,149,Elective,2022-10-15,Aspirin,Inconclusive
42323,AMy GREEN,79,Female,B+,Obesity,2021-03-30,Brett Johnson,Taylor-Williamson,UnitedHealthcare,23402.358491,249,Elective,2021-04-27,Penicillin,Abnormal
50151,AMy GREEN,79,Female,B+,Obesity,2021-03-30,Brett Johnson,Taylor-Williamson,UnitedHealthcare,23402.358491,249,Elective,2021-04-27,Penicillin,Abnormal
21675,ANDREA HansEN,61,Male,O+,Cancer,2021-07-02,Alisha Flores,LLC Clark,Cigna,40026.763948,254,Elective,2021-07-22,Paracetamol,Normal
51695,ANDREA HansEN,61,Male,O+,Cancer,2021-07-02,Alisha Flores,LLC Clark,Cigna,40026.763948,254,Elective,2021-07-22,Paracetamol,Normal
36207,ANDrEA fREnCH,73,Male,A-,Arthritis,2021-02-08,Danny Oconnor,"Taylor Small, Lin and",UnitedHealthcare,9981.590235,323,Elective,2021-02-24,Ibuprofen,Inconclusive
51916,ANDrEA fREnCH,73,Male,A-,Arthritis,2021-02-08,Danny Oconnor,"Taylor Small, Lin and",UnitedHealthcare,9981.590235,323,Elective,2021-02-24,Ibuprofen,Inconclusive


In [6]:
df = df.drop_duplicates().reset_index(drop=True)

duplicate_count_after = df.duplicated().sum()

print("Dataset shape after duplicate removal:", df.shape)
print("Exact duplicate rows after removal:", duplicate_count_after)


Dataset shape after duplicate removal: (54966, 15)
Exact duplicate rows after removal: 0


## 6. Convert Date Columns

Convert admission and discharge dates into pandas datetime values.

The discharge date and derived length of stay are retained only for descriptive analysis in this notebook. They should not be used as input features when making a prediction at the time of admission because they are known only after admission.


In [7]:
date_columns = ["Date of Admission", "Discharge Date"]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

print("Invalid admission dates:", df["Date of Admission"].isna().sum())
print("Invalid discharge dates:", df["Discharge Date"].isna().sum())

display(df[date_columns].head())


Invalid admission dates: 0
Invalid discharge dates: 0


,Date of Admission,Discharge Date
0,2024-01-31,2024-02-02
1,2019-08-20,2019-08-26
2,2022-09-22,2022-10-07
3,2020-11-18,2020-12-18
4,2022-09-19,2022-10-09


## 7. Validate Date Relationships

In [8]:
invalid_date_order = df[
    df["Date of Admission"] > df["Discharge Date"]
]

print("Rows where admission date is after discharge date:",
      len(invalid_date_order))

if len(invalid_date_order) > 0:
    display(invalid_date_order.head())


Rows where admission date is after discharge date: 0


## 8. Create Descriptive Length-of-Stay Feature

Length of stay is calculated for descriptive analysis.

> **Leakage warning:** Length of stay uses the discharge date and must not be used as an input feature for a model that predicts risk at admission.


In [9]:
df["Length of Stay"] = (
    df["Discharge Date"] - df["Date of Admission"]
).dt.days

print("Length-of-stay summary:")
display(df["Length of Stay"].describe())

print("Invalid length-of-stay rows:",
      (df["Length of Stay"] < 0).sum())


Length-of-stay summary:


count    54966.000000
mean        15.499290
std          8.661471
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: Length of Stay, dtype: float64

Invalid length-of-stay rows: 0


## 9. Create Admission-Time Features

The following features are available from the admission date:

- Admission year
- Admission month
- Admission day
- Admission weekday
- Admission quarter

These features are created from the admission date and do not use discharge information.


In [10]:
df["Admission Year"] = df["Date of Admission"].dt.year
df["Admission Month"] = df["Date of Admission"].dt.month
df["Admission Day"] = df["Date of Admission"].dt.day
df["Admission Weekday"] = df["Date of Admission"].dt.dayofweek
df["Admission Quarter"] = df["Date of Admission"].dt.quarter

display(
    df[
        [
            "Date of Admission",
            "Admission Year",
            "Admission Month",
            "Admission Day",
            "Admission Weekday",
            "Admission Quarter",
        ]
    ].head()
)


,Date of Admission,Admission Year,Admission Month,Admission Day,Admission Weekday,Admission Quarter
0,2024-01-31,2024,1,31,2,1
1,2019-08-20,2019,8,20,1,3
2,2022-09-22,2022,9,22,3,3
3,2020-11-18,2020,11,18,2,4
4,2022-09-19,2022,9,19,0,3


## 10. Create Age Groups

Age is grouped into broad categories for exploratory analysis and possible future categorical modeling.

The original numerical `Age` column is retained.


In [11]:
age_bins = [0, 18, 30, 45, 60, 75, 120]
age_labels = [
    "0-17",
    "18-29",
    "30-44",
    "45-59",
    "60-74",
    "75+"
]

df["Age Group"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True
)

print("Age group distribution:")
display(df["Age Group"].value_counts(dropna=False).sort_index())


Age group distribution:


Age Group
0-17       116
18-29     9489
30-44    12132
45-59    12281
60-74    12048
75+       8900
Name: count, dtype: int64

## 11. Review Categorical and Numerical Features

In [12]:
numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)


Numerical columns:
['Age', 'Billing Amount', 'Room Number', 'Length of Stay']

Categorical columns:
['Name', 'Gender', 'Blood Type', 'Medical Condition', 'Doctor', 'Hospital', 'Insurance Provider', 'Admission Type', 'Medication', 'Test Results', 'Age Group']


C:\Users\gousb\AppData\Local\Temp\ipykernel_11524\3013482269.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


## 12. Define Columns for Future Admission-Time Modeling

The following columns are excluded from the admission-time feature set:

- `Name`: personally identifying information and not a generalizable clinical feature.
- `Doctor`: high-cardinality identifier-like field.
- `Hospital`: high-cardinality identifier-like field.
- `Discharge Date`: available after admission.
- `Length of Stay`: derived using discharge information.
- `Test Results`: requires domain review and may reflect information obtained during the hospital stay.
- `Admission Year`: retained in the dataset for analysis but should be reviewed for temporal bias before modeling.
- `Date of Admission`: replaced by derived date features.

This is a preliminary feature-selection decision and should be reviewed after the organizers confirm the target definition and prediction time.


In [13]:
excluded_columns = [
    "Name",
    "Doctor",
    "Hospital",
    "Discharge Date",
    "Length of Stay",
    "Test Results",
    "Date of Admission",
]

admission_time_features = [
    column for column in df.columns
    if column not in excluded_columns
]

print("Candidate admission-time feature columns:")
print(admission_time_features)

modeling_df = df[admission_time_features].copy()

print("\nCandidate modeling dataset shape:", modeling_df.shape)
display(modeling_df.head())


Candidate admission-time feature columns:
['Age', 'Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Medication', 'Admission Year', 'Admission Month', 'Admission Day', 'Admission Weekday', 'Admission Quarter', 'Age Group']

Candidate modeling dataset shape: (54966, 15)


,Age,Gender,Blood Type,Medical Condition,Insurance Provider,Billing Amount,Room Number,Admission Type,Medication,Admission Year,Admission Month,Admission Day,Admission Weekday,Admission Quarter,Age Group
0,30,Male,B-,Cancer,Blue Cross,18856.281306,328,Urgent,Paracetamol,2024,1,31,2,1,30-44
1,62,Male,A+,Obesity,Medicare,33643.327287,265,Emergency,Ibuprofen,2019,8,20,1,3,60-74
2,76,Female,A-,Obesity,Aetna,27955.096079,205,Emergency,Aspirin,2022,9,22,3,3,75+
3,28,Female,O+,Diabetes,Medicare,37909.782410,450,Elective,Ibuprofen,2020,11,18,2,4,18-29
4,43,Female,AB+,Cancer,Aetna,14238.317814,458,Urgent,Penicillin,2022,9,19,0,3,30-44


## 13. Check the Candidate Modeling Dataset

In [14]:
print("Missing values in candidate modeling dataset:")
display(modeling_df.isnull().sum())

print("\nDuplicate rows in candidate modeling dataset:",
      modeling_df.duplicated().sum())

print("\nCandidate modeling dataset data types:")
display(modeling_df.dtypes)


Missing values in candidate modeling dataset:


Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Medication            0
Admission Year        0
Admission Month       0
Admission Day         0
Admission Weekday     0
Admission Quarter     0
Age Group             0
dtype: int64


Duplicate rows in candidate modeling dataset: 0

Candidate modeling dataset data types:


Age                      int64
Gender                     str
Blood Type                 str
Medical Condition          str
Insurance Provider         str
Billing Amount         float64
Room Number              int64
Admission Type             str
Medication                 str
Admission Year           int32
Admission Month          int32
Admission Day            int32
Admission Weekday        int32
Admission Quarter        int32
Age Group             category
dtype: object

## 14. Save Processed Datasets

Two files are saved:

1. `healthcare_feature_engineered_full.csv`
   - Contains the cleaned dataset and engineered features.
   - Useful for analysis and later feature-selection decisions.

2. `healthcare_feature_engineered.csv`
   - Contains the preliminary candidate admission-time features.
   - Does not contain a confirmed readmission target.


In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

full_processed_path = (
    PROCESSED_DIR / "healthcare_feature_engineered_full.csv"
)

df.to_csv(full_processed_path, index=False)
modeling_df.to_csv(PROCESSED_DATA_PATH, index=False)

print("Saved full processed dataset:")
print(full_processed_path.resolve())

print("\nSaved candidate modeling dataset:")
print(PROCESSED_DATA_PATH.resolve())


Saved full processed dataset:
D:\hospital-readmission-project\ml-training\dataset\processed\healthcare_feature_engineered_full.csv

Saved candidate modeling dataset:
D:\hospital-readmission-project\ml-training\dataset\processed\healthcare_feature_engineered.csv


## 15. Final Summary

In [16]:
print("Final full dataset shape:", df.shape)
print("Final candidate modeling dataset shape:", modeling_df.shape)
print("Exact duplicates in full dataset:", df.duplicated().sum())
print("Exact duplicates in candidate dataset:", modeling_df.duplicated().sum())

print("\nNext required step:")
print("Confirm the readmission target and prediction-time requirements before model training.")


Final full dataset shape: (54966, 22)
Final candidate modeling dataset shape: (54966, 15)
Exact duplicates in full dataset: 0
Exact duplicates in candidate dataset: 0

Next required step:
Confirm the readmission target and prediction-time requirements before model training.


## Conclusion

The dataset has been prepared for the next stage:

- Exact duplicate records were investigated and removed.
- Date columns were converted.
- Admission-time date features were created.
- Age groups were created.
- Candidate modeling features were identified.
- Processed datasets were saved.




In [17]:
# Final validation of the candidate modeling dataset

print("Dataset shape:", modeling_df.shape)

print("\nMissing values:")
print(modeling_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(modeling_df.duplicated().sum())

print("\nData types:")
display(modeling_df.dtypes)

print("\nUnique values per column:")
display(modeling_df.nunique().sort_values())

print("\nAge range:")
print(modeling_df["Age"].min(), "to", modeling_df["Age"].max())

print("\nBilling amount range:")
print(
    modeling_df["Billing Amount"].min(),
    "to",
    modeling_df["Billing Amount"].max()
)

Dataset shape: (54966, 15)

Missing values:
0

Duplicate rows:
0

Data types:


Age                      int64
Gender                     str
Blood Type                 str
Medical Condition          str
Insurance Provider         str
Billing Amount         float64
Room Number              int64
Admission Type             str
Medication                 str
Admission Year           int32
Admission Month          int32
Admission Day            int32
Admission Weekday        int32
Admission Quarter        int32
Age Group             category
dtype: object


Unique values per column:


Gender                    2
Admission Type            3
Admission Quarter         4
Insurance Provider        5
Medication                5
Age Group                 6
Admission Year            6
Medical Condition         6
Admission Weekday         7
Blood Type                8
Admission Month          12
Admission Day            31
Age                      77
Room Number             400
Billing Amount        50000
dtype: int64


Age range:
13 to 89

Billing amount range:
-2008.4921398591305 to 52764.276736469175


Investigate negative billing amounts 

In [18]:
# Investigate negative billing amounts

negative_billing = df[df["Billing Amount"] < 0].copy()

print("Number of negative billing records:",
      len(negative_billing))

print("\nNegative billing statistics:")
display(negative_billing["Billing Amount"].describe())

print("\nNegative billing records:")
display(negative_billing.head(20))

Number of negative billing records: 106

Negative billing statistics:


count     106.000000
mean     -502.472497
std       429.857027
min     -2008.492140
25%      -803.004972
50%      -374.840042
75%      -149.074654
max       -23.866729
Name: Billing Amount, dtype: float64


Negative billing records:


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,...,Discharge Date,Medication,Test Results,Length of Stay,Admission Year,Admission Month,Admission Day,Admission Weekday,Admission Quarter,Age Group
132,ashLEy ERIcKSoN,32,Female,AB-,Cancer,2019-11-05,Gerald Hooper,"and Johnson Moore, Branch",Aetna,-502.507813,...,2019-11-23,Penicillin,Normal,18,2019,11,5,1,4,30-44
799,CHRisTOPHer wEiss,49,Female,AB-,Asthma,2023-02-16,Kelly Thompson,Hunter-Hughes,Aetna,-1018.245371,...,2023-03-09,Penicillin,Inconclusive,21,2023,2,16,3,1,45-59
1018,AsHley WaRnER,60,Male,A+,Hypertension,2021-12-21,Andrea Bentley,"and Wagner, Lee Klein",Aetna,-306.364925,...,2022-01-11,Ibuprofen,Normal,21,2021,12,21,1,4,60-74
1421,JAY galloWaY,74,Female,O+,Asthma,2021-01-20,Debra Everett,Group Peters,Blue Cross,-109.097122,...,2021-02-09,Ibuprofen,Abnormal,20,2021,1,20,2,1,60-74
2103,josHUa wilLIamSon,72,Female,B-,Diabetes,2021-03-21,Wendy Ramos,"and Huff Reeves, Dennis",Blue Cross,-576.727907,...,2021-04-17,Aspirin,Abnormal,27,2021,3,21,6,1,60-74
2696,Scott VaZqUEz,74,Male,B+,Diabetes,2023-04-12,Edward Yates,James Ltd,Medicare,-135.986000,...,2023-05-03,Ibuprofen,Abnormal,21,2023,4,12,2,2,60-74
2855,CaROl aNDErSoN,39,Female,B-,Hypertension,2020-04-03,Dr. Patrick Hines,"Carter Carter, and Patterson",Blue Cross,-370.983674,...,2020-04-10,Ibuprofen,Abnormal,7,2020,4,3,4,2,30-44
3772,mr. ChRIStOPhER aLvARaDO,77,Male,AB+,Obesity,2022-06-03,Mr. Dean Guzman DDS,Johnson Inc,Blue Cross,-1310.272895,...,2022-06-13,Paracetamol,Inconclusive,10,2022,6,3,4,2,75+
5445,ALExAndrA KHaN,32,Male,AB+,Arthritis,2022-07-14,Michael Vaughn,"Bowen Lopez, and Terry",Aetna,-692.408820,...,2022-07-19,Lipitor,Abnormal,5,2022,7,14,3,3,30-44
5708,JosEPh cOx,23,Male,AB-,Diabetes,2019-10-13,Peter Smith,Inc Ward,Blue Cross,-353.865186,...,2019-10-25,Lipitor,Inconclusive,12,2019,10,13,6,4,18-29


Then check their distribution

In [19]:
# Review negative billing amounts by relevant categories

display(
    negative_billing[
        [
            "Age",
            "Gender",
            "Medical Condition",
            "Insurance Provider",
            "Billing Amount",
            "Admission Type",
        ]
    ].head(20)
)

,Age,Gender,Medical Condition,Insurance Provider,Billing Amount,Admission Type
132,32,Female,Cancer,Aetna,-502.507813,Urgent
799,49,Female,Asthma,Aetna,-1018.245371,Elective
1018,60,Male,Hypertension,Aetna,-306.364925,Elective
1421,74,Female,Asthma,Blue Cross,-109.097122,Emergency
2103,72,Female,Diabetes,Blue Cross,-576.727907,Urgent
2696,74,Male,Diabetes,Medicare,-135.986000,Elective
2855,39,Female,Hypertension,Blue Cross,-370.983674,Elective
3772,77,Male,Obesity,Blue Cross,-1310.272895,Elective
5445,32,Male,Arthritis,Aetna,-692.408820,Elective
5708,23,Male,Diabetes,Blue Cross,-353.865186,Elective
